# NLP scaling

`pounce` exposes the same two scaling layers as upstream Ipopt:

* `nlp_scaling_method` — `none` / `gradient-based` (default) /
  `user-scaling`. Rescales the objective and each constraint *before*
  the IPM sees them.
* `linear_system_scaling` — `none` (default) / `mc19` / `ruiz` /
  `slack-based`. Symmetric scaling of the KKT augmented system for
  the factorization.

This notebook works through the NLP layer end-to-end. It is the layer
that moves the iteration count on problems with mixed natural units,
and the only layer with a wired Python entry point today. See the
[Scaling reference](https://kitchingroup.cheme.cmu.edu/pounce/scaling.html)
for the full write-up.

In [1]:
import numpy as np
import pounce
print(f"pounce {pounce.__version__}")

pounce 0.9.0


## A constraint scaled by `1e12`

A small two-variable problem: stay as close as possible to
`(3, 2)` while sitting on a circle of radius √13 ≈ 3.606. We multiply
the constraint by `1e12` to simulate a severe unit mismatch (e.g. the
constraint comes from a physical equation expressed in tiny natural
units).

$$
\min_{x \in \mathbb{R}^2}\; (x_1 - 3)^2 + (x_2 - 2)^2
\quad\text{s.t.}\quad 10^{12}\,(x_1^2 + x_2^2 - 13) = 0,\;
-10 \le x_i \le 10.
$$

The optimum is `x* = (3, 2)`. But the constraint Jacobian at any
reasonable starting point is `O(1e13)` — wildly different from the
objective gradient's `O(10)`. Without scaling the IPM still gets
there, but the badly-conditioned KKT system makes it crawl: roughly
**3× the iterations** of the scaled runs. Scaling brings the rows back
into balance and keeps the iteration count flat no matter how large
the constraint's units are.

In [2]:
class HardScaled:
    """min (x1-3)^2 + (x2-2)^2  s.t.  1e12*(x1^2 + x2^2 - 13) = 0."""

    def objective(self, x):
        return (x[0] - 3.0) ** 2 + (x[1] - 2.0) ** 2

    def gradient(self, x):
        return np.array([2.0 * (x[0] - 3.0), 2.0 * (x[1] - 2.0)])

    def constraints(self, x):
        return np.array([1.0e12 * (x[0] ** 2 + x[1] ** 2 - 13.0)])

    def jacobianstructure(self):
        return (np.array([0, 0]), np.array([0, 1]))

    def jacobian(self, x):
        return np.array([2.0e12 * x[0], 2.0e12 * x[1]])

    def hessianstructure(self):
        # 2x2 lower triangle: (0,0), (1,0), (1,1).
        return (np.array([0, 1, 1]), np.array([0, 0, 1]))

    def hessian(self, x, lam, obj_factor):
        # H = obj_factor * 2I + lam[0] * 2e12 * I.
        d = 2.0 * obj_factor + 2.0e12 * lam[0]
        return np.array([d, 0.0, d])


def make(extra=None):
    p = pounce.Problem(
        n=2, m=1, problem_obj=HardScaled(),
        lb=[-10.0, -10.0], ub=[10.0, 10.0],
        cl=[0.0], cu=[0.0],
    )
    p.add_option("tol", 1e-8)
    p.add_option("print_level", 0)
    p.add_option("max_iter", 500)
    for k, v in (extra or {}).items():
        p.add_option(k, v)
    return p


x0 = np.array([-5.0, 4.0])  # start on the opposite side of the circle

## Run 1 — no scaling

Tell the solver to skip its automatic scaling pass.

In [3]:
p = make({"nlp_scaling_method": "none"})
x, info = p.solve(x0=x0)
print(f"status: {info['status_msg']}")
print(f"iters : {info['iter_count']}")
print(f"f*    : {info['obj_val']:.6f}")
print(f"x*    : ({x[0]:.4f}, {x[1]:.4f})")

status: Solve_Succeeded
iters : 34
f*    : 0.000000
x*    : (3.0000, 2.0000)


With no scaling the solve **succeeds but is slow** — it needs far more
iterations than the scaled runs below, because the linear-system
conditioning is dominated by the `1e12` Jacobian. It reaches the right
answer `(3, 2)`; it just spends many extra Newton steps wrestling the
poorly-scaled KKT system into shape. Push the constraint scale higher
(`1e14`, `1e16`) and the unscaled iteration count keeps climbing while
the scaled runs stay flat.

## Run 2 — default `gradient-based` scaling

`pounce`'s default. At the starting point, the objective gradient
∞-norm is `O(10)` (below the `nlp_scaling_max_gradient = 100` cutoff,
so untouched), and the constraint Jacobian ∞-norm is `O(1e13)` (far
above the cutoff). The per-row constraint scale is set to
`100 / 1e13 = 1e-11`, bringing the scaled Jacobian back into the
`O(100)` band — and the iteration count back down to the well-scaled
baseline.

In [4]:
p = make()  # gradient-based is the default
x, info = p.solve(x0=x0)
print(f"status: {info['status_msg']}")
print(f"iters : {info['iter_count']}")
print(f"f*    : {info['obj_val']:.6f}")
print(f"x*    : ({x[0]:.4f}, {x[1]:.4f})")

status: Solve_Succeeded
iters : 12
f*    : 0.000000
x*    : (3.0000, 2.0000)


## Run 3 — `user-scaling`

Suppose we know up front that the constraint is `1e12` larger than the
natural physical relation. We can tell the solver directly via
`Problem.set_problem_scaling(obj_scaling, x_scaling=None,
g_scaling=None)`. This mirrors the C-API `SetIpoptProblemScaling`.

We pair it with `nlp_scaling_method = "user-scaling"` so the IPM
consults the values we just installed instead of computing its own.

In [5]:
p = make({"nlp_scaling_method": "user-scaling"})
p.set_problem_scaling(
    obj_scaling=1.0,
    g_scaling=np.array([1.0e-12]),
)
x, info = p.solve(x0=x0)
print(f"status: {info['status_msg']}")
print(f"iters : {info['iter_count']}")
print(f"f*    : {info['obj_val']:.6f}")
print(f"x*    : ({x[0]:.4f}, {x[1]:.4f})")

status: Solve_Succeeded
iters : 12
f*    : 0.000000
x*    : (3.0000, 2.0000)


## Side-by-side

In [6]:
rows = []
for name, opts in [
    ("none",         {"nlp_scaling_method": "none"}),
    ("gradient",     {"nlp_scaling_method": "gradient-based"}),
    ("user 1e-12",   {"nlp_scaling_method": "user-scaling"}),
]:
    p = make(opts)
    if "user" in name:
        p.set_problem_scaling(1.0, g_scaling=np.array([1.0e-12]))
    x, info = p.solve(x0=x0)
    rows.append((name, info['iter_count'], info['status_msg'], info['obj_val']))

print(f"{'scaling':>12s}  {'iters':>5s}  {'status':<32s}  {'f*':>10s}")
print('-' * 70)
for name, it, status, fval in rows:
    print(f"{name:>12s}  {it:>5d}  {status:<32s}  {fval:.6f}")

     scaling  iters  status                                    f*
----------------------------------------------------------------------
        none     34  Solve_Succeeded                   0.000000
    gradient     12  Solve_Succeeded                   0.000000
  user 1e-12     12  Solve_Succeeded                   0.000000


## Reported quantities stay in natural units

Notice all three runs report `f*` in the same range (objective in the
original units of the TNLP). `pounce` undoes any scaling before
populating `info['obj_val']`, `info['mult_g']`, and the bound
multipliers. Internally, the IPM operates in scaled space: stopping
criteria (`tol`, `acceptable_tol`) compare scaled values, the barrier
parameter μ is in scaled units, and the filter's history is built
from scaled function values.

## Target-gradient overrides

`nlp_scaling_obj_target_gradient` and
`nlp_scaling_constr_target_gradient` are subtle. When set to a
positive value they *override* the `max_gradient` cutoff and the 1.0
clamp: the per-row scale becomes `target / row_max` unconditionally,
so the scaled gradient ∞-norm is pinned exactly to the target. Useful
when you have a specific numeric range in mind.

Default `0.0` means "use the cutoff path" — only scale rows above
`nlp_scaling_max_gradient`.

In [7]:
p = make({
    "nlp_scaling_method": "gradient-based",
    "nlp_scaling_constr_target_gradient": 1.0,  # pin to ∞-norm = 1
})
x, info = p.solve(x0=x0)
print(f"status: {info['status_msg']}")
print(f"iters : {info['iter_count']}")
print(f"f*    : {info['obj_val']:.6f}")

status: Solve_Succeeded
iters : 12
f*    : 0.000000


## Caveat: `linear_system_scaling` is registered but not yet wired

`linear_system_scaling = none | mc19 | ruiz | slack-based` is also
registered. It controls symmetric scaling of the KKT augmented system
for the factorization. At the time of writing each method's per-row
factor computation is unit-tested but the runtime dispatch in
`alg_builder.rs` still passes `None` to the linear solver. Setting
`linear_system_scaling=ruiz` therefore won't change the iteration
count today. The hook is in place; the end-to-end wire-up is tracked
separately from issue #61.

If your problem has a badly-conditioned KKT system *today*, the best
existing lever is `ma57_automatic_scaling=yes` (with `pounce` built
against the HSL `ma57` backend).